In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Simulate new and updated source records
changes = [
    (101, "Amit", "Bengaluru", "amit@example.com", "2026-10-14"),
    (106, "Neha", "Pune", "neha@example.com", "2026-10-14")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "email",
    "source_updated_date"
]

df_changes = spark.createDataFrame(changes, columns)

df_changes = (
    df_changes
    .withColumn(
        "source_updated_date",
        F.to_date("source_updated_date")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

display(df_changes)

customer_id,customer_name,city,email,source_updated_date,ingestion_timestamp
101,Amit,Bengaluru,amit@example.com,2026-10-14,2026-10-22T07:50:28.308Z
106,Neha,Pune,neha@example.com,2026-10-14,2026-10-22T07:50:28.308Z


In [0]:
target_table = "myadlasdevcalaog.silver.customers"

delta_target = DeltaTable.forName(spark, target_table)

(
    delta_target.alias("target")
    .merge(
        df_changes.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="""
            source.source_updated_date >
            target.source_updated_date
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental load completed successfully")

Incremental load completed successfully


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    email,
    source_updated_date
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city,email,source_updated_date
101,Amit,Pune,amit@example.com,2026-10-01


In [0]:
%sql
SELECT customer_id, customer_name, city
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city
101,Amit,Pune


In [0]:
%sql
SELECT customer_id, COUNT(*) AS record_count
FROM myadlasdevcalaog.silver.customers
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,record_count


In [0]:
%sql
SELECT customer_id, customer_name, city, email, source_updated_date
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city,email,source_updated_date
101,Amit,Pune,amit@example.com,2026-10-01


In [0]:
%sql
SELECT customer_id, COUNT(*) AS record_count
FROM myadlasdevcalaog.silver.customers
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,record_count


In [0]:
%sql
SELECT COUNT(*) AS total_customers
FROM myadlasdevcalaog.silver.customers;

total_customers
5


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Keep the newest source record for each customer
window_spec = (
    Window.partitionBy("customer_id")
    .orderBy(
        F.col("source_updated_date").desc_nulls_last(),
        F.col("ingestion_timestamp").desc()
    )
)

df_changes_deduplicated = (
    df_changes
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Incoming source rows:", df_changes.count())
print("Rows after deduplication:", df_changes_deduplicated.count())

display(df_changes_deduplicated)


Incoming source rows: 2
Rows after deduplication: 2


customer_id,customer_name,city,email,source_updated_date,ingestion_timestamp
101,Amit,Bengaluru,amit@example.com,2026-10-14,2026-10-22T07:51:17.650Z
106,Neha,Pune,neha@example.com,2026-10-14,2026-10-22T07:51:17.650Z


In [0]:

target_table = "myadlasdevcalaog.silver.customers"

delta_target = DeltaTable.forName(spark, target_table)

(
    delta_target.alias("target")
    .merge(
        df_changes_deduplicated.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="""
            source.source_updated_date > target.source_updated_date
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental MERGE completed.")


Incremental MERGE completed.


In [0]:
from delta.tables import DeltaTable

In [0]:
%sql
SELECT customer_id, COUNT(*) AS record_count
FROM myadlasdevcalaog.silver.customers
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,record_count


In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM myadlasdevcalaog.silver.customers;

total_rows
5


In [0]:
%sql
SELECT customer_id, customer_name, city
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city
101,Amit,Pune


In [0]:
display(df_changes_deduplicated.select(
    "customer_id",
    "customer_name",
    "city",
    "source_updated_date"
).orderBy("customer_id"))

customer_id,customer_name,city,source_updated_date
101,Amit,Bengaluru,2026-10-14
106,Neha,Pune,2026-10-14


In [0]:
window_spec = (
    Window.partitionBy("customer_id")
    .orderBy(
        F.col("source_updated_date").desc_nulls_last(),
        F.col("ingestion_timestamp").desc()
    )
)

df_changes_deduplicated = (
    df_changes
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

display(df_changes_deduplicated)

customer_id,customer_name,city,email,source_updated_date,ingestion_timestamp
101,Amit,Bengaluru,amit@example.com,2026-10-14,2026-10-22T07:51:18.988Z
106,Neha,Pune,neha@example.com,2026-10-14,2026-10-22T07:51:18.988Z


In [0]:
display(
    df_changes_deduplicated.select(
        "customer_id",
        "customer_name",
        "city",
        "source_updated_date"
    ).orderBy("customer_id")
)

customer_id,customer_name,city,source_updated_date
101,Amit,Bengaluru,2026-10-14
106,Neha,Pune,2026-10-14


In [0]:
target_table = "myadlasdevcalaog.silver.customers"

delta_target = DeltaTable.forName(spark, target_table)

(
    delta_target.alias("target")
    .merge(
        df_changes_deduplicated.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="""
            source.source_updated_date > target.source_updated_date
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental MERGE completed successfully.")

Incremental MERGE completed successfully.


In [0]:
%sql
SELECT customer_id, customer_name, city, source_updated_date
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01


In [0]:
%sql
SELECT COUNT(*) AS total_customers
FROM myadlasdevcalaog.silver.customers;

total_customers
5


In [0]:
target_table = "myadlasdevcalaog.silver.customers"

print("Target table:", target_table)

display(
    spark.table(target_table)
    .select("customer_id", "customer_name", "city", "source_updated_date")
    .orderBy("customer_id")
)

Target table: myadlasdevcalaog.silver.customers


customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01
102,Priya,Mumbai,2026-10-02
103,Rahul,Nashik,2026-10-03
104,Sneha,Pune,2026-10-04
105,Vijay,Nagpur,2026-10-05


In [0]:
print("Source customer IDs:")
display(
    df_changes_deduplicated
    .select("customer_id")
    .orderBy("customer_id")
)

print("Target customer IDs after MERGE:")
display(
    spark.table(target_table)
    .select("customer_id")
    .orderBy("customer_id")
)

Source customer IDs:


customer_id
101
106


Target customer IDs after MERGE:


customer_id
101
102
103
104
105


In [0]:
spark.table("myadlasdevcalaog.silver.customers").printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- source_updated_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
df_changes_deduplicated.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- source_updated_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)



In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "myadlasdevcalaog.silver.customers"

# Check whether customer 106 exists before MERGE
before_count = (
    spark.table(target_table)
    .filter(F.col("customer_id") == 106)
    .count()
)
print("Customer 106 rows before MERGE:", before_count)

delta_target = DeltaTable.forName(spark, target_table)

(
    delta_target.alias("target")
    .merge(
        df_changes_deduplicated.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="""
            source.source_updated_date > target.source_updated_date
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Verify customer 106 immediately
display(
    spark.table(target_table)
    .filter(F.col("customer_id").isin([101, 106]))
    .select("customer_id", "customer_name", "city", "source_updated_date")
    .orderBy("customer_id")
)


Customer 106 rows before MERGE: 0


customer_id,customer_name,city,source_updated_date
101,Amit,Bengaluru,2026-10-14
106,Neha,Pune,2026-10-14


In [0]:
%sql
SELECT customer_id, COUNT(*) AS record_count
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
GROUP BY customer_id
ORDER BY customer_id;

customer_id,record_count
101,1
106,1


In [0]:
%sql
SELECT customer_id, COUNT(*) AS record_count
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
GROUP BY customer_id
ORDER BY customer_id;

customer_id,record_count
101,1
106,1
